# Mutational Robustness Visualizations

Examines the mutational robustness metrics for some populations of matrices

1. The **initial ("stable") population** generated by each version, under matched
   parameters.
2. `compute_mut_robustness` called directly on the *same* matrices under each version,
   to see whether the mutational-robustness computation itself has diverged.

Both implementations are loaded side by side in this session (see the setup cell for how
the old module tree is isolated from the current `src/` files).

In [ ]:
using Random, Distributions, Statistics, Plots

const ROOT_DIR = normpath(joinpath(pwd(), ".."))
const SRC_DIR = joinpath(ROOT_DIR, "src")
const SCRATCH_DIR = mktempdir()

include(joinpath(SRC_DIR, "wagner_algorithm.jl"))
include(joinpath(SRC_DIR, "data_processing.jl"))

const BN = BooleanNetwork
const Stats = CustomStats

println("Packages Loaded!")

## 1. Initial stable population

In [ ]:
Random.seed!(42)

N = 10
weights_dist = Normal(0.0, 1.0)
density = 1.0
pop_size = 200

params = BN.SimulationParameters(
    number_genes=N, pop_size=pop_size, initial_pop_type=:stable,
    initial_density=density, weights_dist=weights_dist
)
initial_state, optimal_phenotype, matrices = BN.initialize_population(params)

In [ ]:
nonzero_stats(M) = begin
    nz = filter(!iszero, vec(M))
    isempty(nz) ? (0.0, 0.0, 0.0) : (length(nz) / length(M), mean(nz), std(nz))
end

density = [nonzero_stats(M)[1] for M in matrices]
mean_w  = [nonzero_stats(M)[2] for M in matrices]
std_w   = [nonzero_stats(M)[3] for M in matrices]

println("Density         - $(mean(density)) +/- $(std(density))")
println("Mean of weights - $(mean(mean_w))  +/- $(std(mean_w))")
println("Std of weights  - $(mean(std_w))   +/- $(std(std_w))")

p1 = histogram(mean_w, bins=30, title="Per-matrix mean weight", color=:cornflowerblue)

p2 = histogram(std_w, bins=30, title="Per-matrix weight std", color=:mediumpurple3)

plot(p1, p2, layout=(1, 2), size=(900, 350))

## 2. `compute_mut_robustness` Distributions

In [ ]:
Random.seed!(123)

n_trials = 100
n_mutations = 20      # matches Experiment1Config.number_mutation_samples default
n_noise_masks = 50   # matches Experiment1Config.number_noise_samples default
noise_dist = Gamma(1.0, 1.0)  # matches Experiment1Config.standard_noise default
mutation_prob = 0.01          # matches STANDARD_PARAMETERS["pr"] / mutation_prob default

shift = Vector{Float64}(undef, n_trials)
var   = Vector{Float64}(undef, n_trials)
prob_shift = Vector{Float64}(undef, n_trials)
prob_var = Vector{Float64}(undef, n_trials)

for i in 1:n_trials
    M = matrices[i]

    result = Stats.compute_mut_robustness(
        M, initial_state, n_mutations, n_noise_masks, noise_dist, mutation_prob;
        weights_dist=weights_dist, max_steps=params.max_steps,
        activation_function=params.activation_function,
        development_type=params.development_type
    )
    shift[i] = result.stable_expression_shift
    var[i]   = result.stable_expression_var
    prob_shift[i] = result.unstable_prob_shift
    prob_var[i] = result.unstable_prob_var
end

println("stable_expression_shift - $(mean(shift)) +/- $(std(shift)) (max $(maximum(shift)))")
println("stable_expression_var   - $(mean(var)) +/- $(std(var)) (max $(maximum(var)))")
println("unstable_prob_shift - $(mean(prob_shift)) +/- $(std(prob_shift)) (max $(maximum(prob_shift)))")
println("unstabñe_prob_var   - $(mean(prob_var)) +/- $(std(prob_var)) (max $(maximum(prob_var)))")

In [ ]:
p3 = histogram(shift, bins=30, title="stable_expression_shift", color=:lightpink3)
p4 = histogram(var, bins=30, title="stable_expression_var", color=:lightpink1)

plot(p3, p4, layout=(1, 2), size=(900, 350))

In [ ]:
p5 = histogram(prob_shift, bins=30, title="unstable_probability_shift", color=:palegreen3)
p6 = histogram(prob_var, bins=30, title="unstable_probability_variance", color=:darkolivegreen2)

plot(p5, p6, layout=(1, 2), size=(900, 350))